# Part4
# GAN-BERT, G2 Architecture

In [1]:
# If we want to run the code in kaggle, we need this!

!pip install gdown

In [2]:
# Import libraries
import json

import tqdm
import torch
import random

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModel, AutoTokenizer, AutoConfig
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

In [3]:
# Set seed
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_val)

In [4]:
# We use this function for changing the device
def to_device(data, device):
    # for every batch, we pass the data to the current device.
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)


# We use this class for passing the data and model to the device
class DeviceDataLoader():
    def __init__(self, dl, device):
        self.dl=dl
        self.device=device

    def __iter__(self):
        #For every batch, we pass it to the current device.
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        return len(self.dl)

# Selecting the current device
if torch.cuda.is_available():
      device=torch.device('cuda')
else:
      device=torch.device('cpu')

print("The device is:",device)


The device is: cuda


In [5]:
# Load JSON files
# !gdown 1oh9c-d0fo3NtETNySmCNLUc6H1j4dSWE
# !gdown 1k5LMwmYF7PF-BzYQNE2ULBae79nbM268

!gdown 1LFeGWL49PX5JujrQ2zMbSgxuAFA-Xmbf
!gdown 1SZNvp_hYVczqe0rM1AZKu9tXTE7qHN1j

# Read train data and make list of texts and their labels
all_texts_train=[]
all_labels_train=[]
with open('subtaskB_train.jsonl','r') as f:
     for line in f:
        data = json.loads(line)
        all_texts_train.append(data['text'])
        all_labels_train.append(data['model'])

# Read test/val data and make list of texts and their labels
all_texts_test=[]
all_labels_test=[]
lennns=[]
with open('subtaskB_dev.jsonl','r') as f:
    for line in f:
        data = json.loads(line)
        all_texts_test.append(data['text'])
        all_labels_test.append(data['model'])
        lennns.append(len(data['text']))

Downloading...
From: https://drive.google.com/uc?id=1LFeGWL49PX5JujrQ2zMbSgxuAFA-Xmbf
To: /kaggle/working/subtaskB_dev.jsonl
100%|███████████████████████████████████████| 4.93M/4.93M [00:00<00:00, 189MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1SZNvp_hYVczqe0rM1AZKu9tXTE7qHN1j
From (redirected): https://drive.google.com/uc?id=1SZNvp_hYVczqe0rM1AZKu9tXTE7qHN1j&confirm=t&uuid=839e5408-b4a5-4e79-9c72-adbce401246d
To: /kaggle/working/subtaskB_train.jsonl
100%|█████████████████████████████████████████| 155M/155M [00:01<00:00, 140MB/s]


In [6]:
# Convert to dataframes
df_train = pd.DataFrame({"Text": all_texts_train, "Label": all_labels_train})
df_test = pd.DataFrame({"Text": all_texts_test, "Label": all_labels_test})

# Prtint Data Stats
print('Number of datapoints in each class:')
print('Train set:')
print(df_train['Label'].value_counts())
print('*' * 30)
print('Test set:')
print(df_test['Label'].value_counts())
print('*' * 30)
print('Train set Shape:',df_train.shape)
print('Test set Shape:',df_test.shape)

Number of datapoints in each class:
Train set:
Label
davinci    11999
bloomz     11998
human      11997
chatGPT    11995
dolly      11702
cohere     11336
Name: count, dtype: int64
******************************
Test set:
Label
chatGPT    500
human      500
davinci    500
cohere     500
bloomz     500
dolly      500
Name: count, dtype: int64
******************************
Train set Shape: (71027, 2)
Test set Shape: (3000, 2)


In [7]:
# Dataset parameters
max_seq_length = 256
batch_size = 16

# Use Percentage% of the labeled data for training
Percentage = 0.5
df_train_for_ganbert = df_train.sample(frac = Percentage)

# Use the (1 - Percentage) remaining as unlabeled
df_unlabeled = df_train.drop(df_train_for_ganbert.index)

# Print labeled and unlabeled datasets shape
print('Labeled Data:',df_train_for_ganbert.shape)
print('Unlabeled Data:',df_unlabeled.shape)

# available labels in dataset
label_list = ['UNK', 'chatGPT', 'human', 'cohere', 'davinci', 'bloomz', 'dolly']

# Set  unknowne label for unlabeled data
for i in df_unlabeled.index :
    df_unlabeled.at[i, "Label"]= "UNK"

Labeled Data: (35514, 2)
Unlabeled Data: (35513, 2)


In [8]:
# Define a label map dictianary for creating datasets
label_map = {}
for (i, label) in enumerate(label_list):
    label_map[label] = i


# A function for get datapoints from df
def get_datapoints(df):
    # A list to store the datapoints
    rows = []
    # Loop through rows
    for _, row in df.iterrows():
        rows.append((row['Text'], row['Label']))
    return rows

# Apply the function and create examples from dfs
labeled_data = get_datapoints(df_train_for_ganbert)
unlabeled_data = get_datapoints(df_unlabeled)
test_data = get_datapoints(df_test)

In [9]:
print('Number of Labeled Datapoints:',len(labeled_data))
print('Number of Unlabeled Datapoints:',len(unlabeled_data))
print('Number of Test Datapoints:',len(test_data))

Number of Labeled Datapoints: 35514
Number of Unlabeled Datapoints: 35513
Number of Test Datapoints: 3000


In [10]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [11]:
# Main function for creating semi-supervised dataset
def create_dataset(input_examples, label_masks, label_map):

    # A list to save data and masks
    the_data = []

    # Loop through the datapoints and get data and the mask
    for idx, data in enumerate(input_examples):
        the_data.append((data, label_masks[idx]))

    input_ids = []
    attention_mask = []
    label_mask_array = []
    label_id_array = []

    # Tokenization
    for (text, label_mask) in tqdm.tqdm(the_data):
        encoded_sent = tokenizer.encode(
            text[0], add_special_tokens=True, max_length=max_seq_length,
            padding="max_length", truncation=True)

        input_ids.append(encoded_sent)
        label_id_array.append(label_map[text[1]])
        label_mask_array.append(label_mask)

    # Attention to token
    for sent in input_ids:
        att_mask = [int(token_id > 0) for token_id in sent]
        attention_mask.append(att_mask)

    # Convert to Tensor
    input_ids = torch.tensor(input_ids)
    attention_mask = torch.tensor(attention_mask)
    label_id_array = torch.tensor(label_id_array, dtype=torch.long)
    label_mask_array = torch.tensor(label_mask_array)

    # Make the TensorDataset
    dataset = TensorDataset(input_ids, attention_mask, label_id_array, label_mask_array)


    return dataset

In [12]:
# Create a semisupervised train datapoints
all_train_data = labeled_data + unlabeled_data

# The labeled dataset has mask = True
train_label_masks = np.ones(len(labeled_data), dtype=bool)

# The unlabeled dataset has mask = False
train_unlabel_masks = np.zeros(len(unlabeled_data), dtype=bool)

# Concat the masks
train_label_masks = np.concatenate([train_label_masks,train_unlabel_masks])

# Create trainset and
train_dataset = create_dataset(all_train_data, train_label_masks, label_map)

# Create train dataloader
train_dataloader = DataLoader(train_dataset, sampler = RandomSampler(train_dataset),batch_size = batch_size)
train_dataloader=DeviceDataLoader(train_dataloader,device)

# The labeled dataset has mask = True
test_label_masks = np.ones(len(test_data), dtype=bool)

# Create testset
test_dataset = create_dataset(test_data, test_label_masks, label_map)

# Create test dataloader
test_dataloader = DataLoader(test_dataset, sampler = SequentialSampler(test_dataset),batch_size = batch_size)
test_dataloader=DeviceDataLoader(test_dataloader,device)

100%|██████████| 71027/71027 [02:06<00:00, 559.87it/s] 
/tmp/ipykernel_107/3942690758.py:35: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  label_mask_array = torch.tensor(label_mask_array)
100%|██████████| 3000/3000 [00:04<00:00, 689.00it/s] 


In [13]:
# Dataloader size
print(f'Number of training batchs with size of {batch_size}:',len(train_dataloader))

Number of training batchs with size of 16: 4440


In [14]:
# Create Generator class
class Generator(nn.Module):
    def __init__(self, bert_model, bag_of_words_dim):
        super(Generator, self).__init__()
        self.bert_model = bert_model
        # self.fc_bow = nn.Linear(bag_of_words_dim, bert_model.config.hidden_size)
        # self.fc_out = nn.Linear(bert_model.config.hidden_size, bert_model.config.vocab_size)

    def forward(self, bag_of_words):

        #bow_embedding = self.fc_bow(bag_of_words)


        generated_output=self.bert_model(bag_of_words[0], attention_mask=bag_of_words[1])


        #output = self.fc_out(generated_output.last_hidden_state)

        return generated_output

class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_layer):
        super(Discriminator, self).__init__()

        self.Discriminator_Features = nn.Sequential(
            nn.Dropout(p=0.1),
            nn.Linear(input_dim, hidden_layers[0]),
            nn.BatchNorm1d(hidden_layers[0]),
            nn.LeakyReLU(0.2),
            nn.Dropout(p=0.1),
        )

        self.GRU = nn.GRU(hidden_layers[0], hidden_layers[1], batch_first=True)

        self.last_linear = nn.Linear(hidden_layers[1], output_layer)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        features = self.Discriminator_Features(x)

        gru_input = features.unsqueeze(1)  # Adding time dimension

        gru_output, _ = self.GRU(gru_input)

        gru_output = gru_output[:, -1, :]  # Taking the last output

        last_linear_val = self.last_linear(gru_output)

        output = self.softmax(last_linear_val)

        return features, last_linear_val, output


In [15]:
# Set hidden size for G/D
hidden_size = 512
hidden_layers_generator = [hidden_size,hidden_size//2, hidden_size//2, hidden_size]
hidden_levels_discriminator = [hidden_size,hidden_size//2, hidden_size//2, hidden_size]

# Set noise and output dimansions
output_layer = 768
bag_of_words_dim=output_layer
# Create G/D models

bert_model_for_generator = AutoModel.from_pretrained('bert-base-uncased')

generator_network = Generator(bert_model=bert_model_for_generator, bag_of_words_dim=bag_of_words_dim)
# We have len(np.unique(all_labels_test)) classes for bert_model, and have 1 label for G and 1 label for unknown data.

discriminator_network = Discriminator( input_dim = output_layer,hidden_layers = hidden_levels_discriminator,output_layer = len(np.unique(all_labels_test)) + 2)


# Load BERT
bert_model = AutoModel.from_pretrained("bert-base-uncased")

# Put everything in the GPU
generator_network=to_device(generator_network,device)
discriminator_network=to_device(discriminator_network,device)
bert_model=to_device(bert_model,device)


In [16]:
# Print G/D architectures
print(generator_network.parameters)
print('*' * 50)
print(discriminator_network.parameters)
print('*' * 50)
print(bert_model.parameters)

<bound method Module.parameters of Generator(
  (bert_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [17]:
# Set number of epochs
num_train_epochs = 3

# Save training/val/test stats
training_stats = []

# List of  models parameters for passing to optimizer
bert_params = [i for i in bert_model.parameters()]
discriminator_params = [v for v in discriminator_network.parameters()] + bert_params
generator_params = [v for v in generator_network.parameters()]

# Set optimization parameters
learning_rate_discriminator = 5e-5
learning_rate_generator = 5e-5
epsilon = 1e-8


def bag_of_words_among_dataset(train_dataset,batch_size,max_length):

  train_loader2 = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  train_loader2 = DeviceDataLoader(train_loader2, device)

  all_input_ids=torch.zeros((batch_size,max_length)).to(device).to(torch.long)
  all_attention_mask=torch.zeros((batch_size,max_length)).to(device).to(torch.long)

  counter=0
  for batch in train_loader2:

    counter=counter+1
    input_ids, attention_mask, _, _ = batch

    random_idx = torch.randint(0,input_ids.shape[1],(1,)).to(device).squeeze()
    all_input_ids[:,counter-1]=input_ids[:,random_idx]
    all_attention_mask[:,counter-1]=attention_mask[:,random_idx]

    if(counter==batch_size):
      break


  return all_input_ids, all_attention_mask

# Set optimizers
discriminator_optimizer = torch.optim.AdamW(discriminator_params, lr=learning_rate_discriminator)
generator_optimizer = torch.optim.AdamW(generator_params, lr=learning_rate_generator)

# Loop through epochs
for epoch in range(0, num_train_epochs):



    train_loss_generator = 0
    train_loss_discriminator = 0

    # Set training mode
    bert_model.train()
    generator_network.train()
    discriminator_network.train()


    print("#################")
    print("This model is working for the train for the epoch",epoch+1)
    print("\n\n\n\n")


    # Loop through the batches
    for step, batch in tqdm.tqdm(enumerate(train_dataloader)):


        # Unpack the training batch
        input_ids, attention_mask, labels, masks = batch


        the_batch_size = input_ids.shape[0]

        # Encode real data in the BERT
        bert_out = bert_model(input_ids, attention_mask=attention_mask)
        feaures_bert = bert_out[-1]

        # Create noise to feed to the generator


        bag_of_words = bag_of_words_among_dataset(train_dataset,batch_size,input_ids.shape[1])


        # Pass bag of words through the BagOfWordsGenerator
        output_generator = generator_network(bag_of_words)



        #Generator2(bert_model=bert_model, bag_of_words_dim=bag_of_words_dim, noise_dim=noise_dim)
        features_generator=torch.squeeze(output_generator[0][:, -1, :])

        # Feed the output of the bert and the generator to disciminator
        disciminator_input = torch.cat([feaures_bert, features_generator], dim=0)

        # Get output of the disciminator
        features, logits, probs = discriminator_network(disciminator_input)

        # Separate the output of discriminatorfor the real and fake
        features_list = torch.split(features, the_batch_size)
        D_real_features = features_list[0]
        D_fake_features = features_list[1]

        logits_list = torch.split(logits, the_batch_size)
        D_real_logits = logits_list[0]
        D_fake_logits = logits_list[1]

        probs_list = torch.split(probs, the_batch_size)
        D_real_probs = probs_list[0]
        D_fake_probs = probs_list[1]

        # Generator LOSS
        g_loss_d = -1 * torch.mean(torch.log(1 - D_fake_probs[:,-1] + epsilon))

        g_feat_reg = torch.mean(torch.pow(
            torch.mean(D_real_features, dim=0)
             - torch.mean(D_fake_features, dim=0), 2))

        all_losses_for_G = g_loss_d + g_feat_reg

        # Disciminator LOSS
        logits = D_real_logits[:,0:-1]
        log_probs = F.log_softmax(logits, dim=-1)

        # The Loss for unlabeled data is masked out
        label2one_hot = torch.nn.functional.one_hot(labels, len(label_list))

        per_example_loss = -torch.sum(label2one_hot * log_probs, dim=-1)

        per_example_loss = torch.masked_select(per_example_loss, masks.to(device))

        labeled_example_count = per_example_loss.type(torch.float32).numel()

        if labeled_example_count == 0:
            D_L_Supervised = 0

        else:
            D_L_Supervised = torch.div(
              torch.sum(per_example_loss.to(device)), labeled_example_count)

        D_L_unsupervised1U = -1 * torch.mean(
            torch.log(1 - D_real_probs[:, -1] + epsilon))

        D_L_unsupervised2U = -1 * torch.mean(
            torch.log(D_fake_probs[:, -1] + epsilon))

        all_losses_for_D= D_L_Supervised + D_L_unsupervised1U + D_L_unsupervised2U

        # Reset gradients
        generator_optimizer.zero_grad()
        discriminator_optimizer.zero_grad()

        # Backward pass
        all_losses_for_G.backward(retain_graph=True)
        all_losses_for_D.backward()

        # Update weights
        generator_optimizer.step()
        discriminator_optimizer.step()

        # Save the losses to report
        train_loss_generator += all_losses_for_G.item()
        train_loss_discriminator += all_losses_for_D.item()

        if (step%20==0):
          print("step",step,"from",1+len(train_dataset)//batch_size)


    # Calculate the average loss over the batches.
    avg_train_loss_g = train_loss_generator / len(train_dataloader)
    avg_train_loss_d = train_loss_discriminator / len(train_dataloader)


    print("Average training generetor loss:",avg_train_loss_g)
    print("Average training discriminator loss:",avg_train_loss_d)


    print("#################")
    print("This model is working for the test for the epoch",epoch+1)
    print("\n\n\n\n")


    # Set validation mode
    bert_model.eval()
    discriminator_network.eval()
    generator_network.eval()

    # Tracking variables
    total_test_accuracy = 0

    total_test_loss = 0
    nb_test_steps = 0

    all_preds = []
    all_labels_ids = []

    # Define Loss function
    nll_loss = torch.nn.CrossEntropyLoss(ignore_index=-1)

    # Loop through test patches
    for batch in test_dataloader:

        input_ids, attention_mask, labels, _ = batch
        # No grad block
        with torch.no_grad():
            bert_out = bert_model(input_ids, attention_mask=attention_mask)

            features_bert = bert_out[-1]

            _, logits, probs = discriminator_network(features_bert)

            filtered_logits = logits[:,0:-1]

            # Accumulate the test loss.
            total_test_loss += nll_loss(filtered_logits, labels)

        # Accumulate the predictions and the input labels
        _, preds = torch.max(filtered_logits, 1)
        all_preds += preds.detach().cpu()
        all_labels_ids += labels.detach().cpu()

    # Report the final accuracy for this validation run.
    all_preds = torch.stack(all_preds).numpy()
    all_labels_ids = torch.stack(all_labels_ids).numpy()
    test_accuracy = np.sum(all_preds == all_labels_ids) / len(all_preds)
    print("Test Accuracy for epoch",epoch+1," is: ",test_accuracy)

    # Calculate the average loss over all of the batches.
    avg_test_loss = total_test_loss / len(test_dataloader)
    avg_test_loss = avg_test_loss.item()

    # Print validation stats
    print("Average training generetor loss:",avg_train_loss_g)


    # Store stats from this epoch.
    training_stats.append({'epoch': epoch + 1,'Test Accuracy': test_accuracy})

#################
This model is working for the train for the epoch 1







1it [00:02,  2.02s/it]

step 0 from 4440


21it [00:27,  1.25s/it]

step 20 from 4440


...

step 4400 from 4440


4421it [1:32:20,  1.25s/it]

step 4420 from 4440


4440it [1:32:44,  1.25s/it]


Average training generetor loss: 0.7377818459191838
Average training discriminator loss: 0.966358803964413
#################
This model is working for the test for the epoch 3





Test Accuracy for epoch 3  is:  0.549
Average training generetor loss: 0.7377818459191838


In [18]:
# Save training and validation/test stats to report later
# and comprasion with other parts

# Write stats as a json file
with open("Part4_stats_50persent_maxlen256.json", "w") as outfile:
     json.dump(training_stats, outfile)
print('Training and testing procedures have been finished completely!')

Training and testing procedures have been finished completely!
